In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from IPython.display import Audio, display
import pandas as pd
from google import genai
from google.genai import types

from multimodal_lancedb import *
from utils import *
from judge import *
from prompt import *

In [2]:
# Initialize the system
search_system = MusicSearchSystem(db_path="./.lancedb_2", music_dir="music")

In [3]:
api_key = os.getenv('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
video_path = 'video/'
input = client.files.upload(file=f"{video_path}20. fashion_video.mp4")
#input = client.files.upload(file="https://storage.cloud.google.com/video-05/1.%20travel_video.mp4")

In [11]:
# generate summary for video
response = client.models.generate_content(
    model="gemini-2.0-flash",
    config=types.GenerateContentConfig(
                system_instruction=SYS_SUMMARY_PROMPT_VIDEO,
                temperature=0.7
    ),
    contents=[USER_PROMPT_VIDEO, input]
)

video_summary = response.text

video_summary

"This clip feels like a high-fashion runway show, demanding a sophisticated and confident musical backdrop. A minimalist electronic track with a pulsing synth bassline could underscore the model's assured walk, creating a sense of modern elegance. Alternatively, a jazzy, downtempo piece with brushed drums and a muted trumpet could lend a cool, understated vibe."

In [12]:
# Search the music from query
retrieval_results = search_system.search_music(video_summary, top_k=200)

# Show explanation of LLM
df_recommendations = pd.DataFrame(retrieval_results["final_rerank"])
df_recommendations.head(5)
# print("\nLLM explanation：")
# print(results['explanation'])

# play music
# print("\nOverlapping Music：")
# for audio_path in results['audio_paths']:
#     print(f"\nNow playing: {os.path.basename(audio_path)}")
#     display(Audio(audio_path))

[2025-06-02T00:15:40Z WARN  lance::dataset] No existing dataset at /home/tinglin/1125_env/experiment/.lancedb_temp/rerank_tmp.lance, it will be created


,song_name,artist,description,similarity_score,similarity_audio,similarity_text,source,rerank_score,audio_path
0,Fashion House Loop,Sawtooz,A powerful and energetic house track with a ca...,0.832679,0.723754,0.879361,"audio,text",0.213541,music/Sawtooz - Fashion House Loop.mp3
1,Event Ceremony Logo,Artlist Musical Logos,"This is a powerful, energetic and dynamic roya...",0.843332,0.801977,0.861056,"audio,text",0.137185,music/Artlist Musical Logos - Event Ceremony L...
2,Sharp Knives - Instrumental Version,Damon Power,A dark and intense trap instrumental with a fu...,0.825747,0.713593,0.873814,"audio,text",0.120232,music/Damon Power - Sharp Knives - Instrumenta...
3,Freaky Boy,Damon Power,"A trendy, upbeat, and energetic hip-hop instru...",0.851348,0.782755,0.880744,"audio,text",0.108945,music/Damon Power - Freaky Boy.mp3
4,The Happy Intro,Korolkov,"A modern, upbeat and stylish electronic track ...",0.912445,0.862492,0.933853,"audio,text",0.103749,music/Korolkov - The Happy Intro.mp3


Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))


In [13]:
# random to choose 5 songs for comparison
df_for_random = pd.read_csv('final_dataset.csv')
sampled_rows = df_for_random[['song_name', 'artist']].sample(5)

song_randoms = []
for _, row in sampled_rows.iterrows():
    path = find_file_path(row['artist'], row['song_name'])
    if path:
        song_randoms.append(path)
    else:
        print(f"File not found: {row['artist']} - {row['song_name']}")

song_randoms

['music/PaBlikMM - Personal Hero Intro Trailer Opener.mp3',
 'music/The Mind Sweepers - Rancid Life - Short Version C.mp3',
 'music/Eugene_Barduja - For Kids.wav',
 'music/Difourks - Energy Dance Loop.wav',
 'music/Alon Ohana - Mercury - Short Version.mp3']

In [14]:
top_1 = df_recommendations.head(1)['audio_path'].iloc[0]
top_5 = df_recommendations.head(5)['audio_path'].iloc[4]
print(top_1, top_5)

music/Sawtooz - Fashion House Loop.mp3 music/Korolkov - The Happy Intro.mp3


In [15]:
music_top1 = client.files.upload(file=top_1)

In [16]:
PAIR_PROMPT = SYS_PAIR_PROMPT
PAIR_PROMPT += f"\n\nHere is the pairing rule:\n{OVERALL_SCORE_PAIRING_PROMPT}"
print(PAIR_PROMPT)


You are a loyal judge, your task is to choose the better one from two responses on the given task. You will be given a task, including the input and the two responses. The pairing rule will also be given, you need to choose with your careful consideration. Judge task require multi-modal inputs, you should use your visual and auditory senses to judge. You should entirely understand, see or hear the task and the response, base on the given information, you should think of your choosing reasons in the each rubric’s "comment" step by step first, and then you are required to give a choice in "choice" base on the rule.
**Choosing Rule:**
Reasoning in detail before you determine the choice, then give your choice from [0,1,2], 0 means the first response is better, 1 means the two responses are equally good, 2 means the second response is better.


Here is the pairing rule:

You are going to choose base on the overall quality of the reponse's performance on the given task.
Overall Quality Defi

In [17]:
# the model to use
# GEMINI_2_FLASH = "gemini-2.0-flash"
# GEMINI_1_5_PRO = "gemini-1.5-pro"
# GEMINI_2_5_PRO = "gemini-2.5-pro-preview-05-06" #Pre-release version
pair_results = []
for audio2 in song_randoms:
    random = client.files.upload(file = audio2)
    pair = model_pair_content('video', input, music_top1, random)
    votes, comments = run_vote(
        client, 
        model_version = GEMINI_2_5_PRO,
        pair_content = pair, 
        sys_prompt = PAIR_PROMPT, 
        n = 1)
    
    pair_results.append({
        "audio1": music_top1,
        "audio2": random,
        "votes": votes,
        "comments": comments
    })
pair_results

[{'audio1': File(name='files/rwvwxe1s7df9', display_name=None, mime_type='audio/mpeg', size_bytes=1203769, create_time=datetime.datetime(2025, 6, 2, 0, 15, 57, 901587, tzinfo=TzInfo(UTC)), expiration_time=datetime.datetime(2025, 6, 4, 0, 15, 57, 675813, tzinfo=TzInfo(UTC)), update_time=datetime.datetime(2025, 6, 2, 0, 15, 57, 901587, tzinfo=TzInfo(UTC)), sha256_hash='YjNhYTEzMzhhMDZkNGM0NmMwZjk0MjMzNGQzMWY1MWUxYzZhMzE1MDRlNjBkNjBjMzQzYjMzNGQ3MmU2MzllOQ==', uri='https://generativelanguage.googleapis.com/v1beta/files/rwvwxe1s7df9', download_uri=None, state=<FileState.ACTIVE: 'ACTIVE'>, source=<FileSource.UPLOADED: 'UPLOADED'>, video_metadata=None, error=None),
  'audio2': File(name='files/9glhpdhlfv7b', display_name=None, mime_type='audio/mpeg', size_bytes=1218379, create_time=datetime.datetime(2025, 6, 2, 0, 16, 4, 594001, tzinfo=TzInfo(UTC)), expiration_time=datetime.datetime(2025, 6, 4, 0, 16, 4, 470707, tzinfo=TzInfo(UTC)), update_time=datetime.datetime(2025, 6, 2, 0, 16, 4, 594001, 

In [18]:
result_list = []

for r in pair_results:
    winner, summary = majority_vote(r['votes'])
    result_list.append({
        'audio1_uri': r['audio1'].uri,
        'audio2_path': r['audio2'],
        'vote_summary': summary,
        'winner': winner,
        'comment': r['comments'][0] if r['comments'] else ''
    })

result_overall = pd.DataFrame(result_list)
result_overall

,audio1_uri,audio2_path,vote_summary,winner,comment
0,https://generativelanguage.googleapis.com/v1be...,name='files/9glhpdhlfv7b' display_name=None mi...,{'2': 1},2,The video depicts a model confidently walking ...
1,https://generativelanguage.googleapis.com/v1be...,name='files/bgo9amlpv8ya' display_name=None mi...,{'0': 1},0,"The first music track, which is electronic dan..."
2,https://generativelanguage.googleapis.com/v1be...,name='files/7zm7rubka6kc' display_name=None mi...,{'0': 1},0,"The first response, with its electronic and up..."
3,https://generativelanguage.googleapis.com/v1be...,name='files/1yvwn43qw6wy' display_name=None mi...,{'0': 1},0,The video shows a model walking on a runway. T...
4,https://generativelanguage.googleapis.com/v1be...,name='files/u1aq8jil71ys' display_name=None mi...,{'2': 1},2,The video depicts a model walking on a runway ...


In [19]:
result_overall['comment'].tolist()

["The video depicts a model confidently walking down a runway. The lighting is dramatic, and the model's expression is serious and focused. Response 1 provides an upbeat, generic electronic dance track that feels out of place and doesn't enhance the mood of the video. Response 2, on the other hand, offers a rock track with a strong, driving beat and a slightly edgy feel. This music better complements the model's confident stride and the overall powerful and stylish atmosphere of a fashion show. The guitar riffs and the build-up in the music create a sense of drama and energy that aligns well with the visuals.",
 "The first music track, which is electronic dance music, is more suitable for the fashion runway video. Its tempo and style align better with the visuals of a model walking, and this genre is commonly used in fashion shows. The second track, a rock piece, feels out of place and doesn't complement the elegant and modern aesthetic of the video.",
 "The first response, with its el

In [20]:
music_top5 = client.files.upload(file=top_5)
top1_5_pair = model_pair_content('video', input, music_top1, music_top5)
top1_5_pair

['Here is the query of video to music retrieval task:\nThis is a/an video . Please evaluate the following two background music based on this video. Which one is more suitable?',
 File(name='files/zvjct2z888h7', display_name=None, mime_type='video/mp4', size_bytes=5783833, create_time=datetime.datetime(2025, 6, 2, 0, 14, 56, 832557, tzinfo=TzInfo(UTC)), expiration_time=datetime.datetime(2025, 6, 4, 0, 14, 56, 707714, tzinfo=TzInfo(UTC)), update_time=datetime.datetime(2025, 6, 2, 0, 14, 56, 832557, tzinfo=TzInfo(UTC)), sha256_hash='NjZmYjk1Y2U4ZDUzMDg0ZTA5MWZmYWQzYjY2M2U2N2NjOTYzYTU0YmFlNWRkYzc2ZjBhZWFlZDYwYmNkYjY1Nw==', uri='https://generativelanguage.googleapis.com/v1beta/files/zvjct2z888h7', download_uri=None, state=<FileState.PROCESSING: 'PROCESSING'>, source=<FileSource.UPLOADED: 'UPLOADED'>, video_metadata=None, error=None),
 '\nHere is the first reponse:\n',
 File(name='files/rwvwxe1s7df9', display_name=None, mime_type='audio/mpeg', size_bytes=1203769, create_time=datetime.datetim

In [21]:
votes_top5, comments_top5 = run_vote(client, GEMINI_2_5_PRO, top1_5_pair, PAIR_PROMPT, 5)

In [22]:
final_decision, vote_summary = majority_vote(votes_top5)

print(f"最終結果：模型 {final_decision} 勝出")
print("投票統計：", vote_summary)
print("評語範例：")
for c in comments_top5:
    print("-", c)

最終結果：模型 0 勝出
投票統計： {'0': 3, '2': 2}
評語範例：
- The first music option is much more suitable for the video. The video depicts a model walking on a runway in a serious, professional, and somewhat intense manner. The first music track is electronic, has a strong, driving beat, and a modern, chic vibe that is commonly associated with fashion shows. It enhances the confident stride of the model. The second music option, while also electronic and upbeat, has a slightly more generic, almost 
- The video shows a model walking on a runway. Response 1 is a generic, upbeat electronic track that feels a bit too much like a party or club scene, not quite fitting the slightly serious and focused mood of a runway. Response 2, while also electronic, has a more driving, stylish, and slightly edgy feel that is much more commonly associated with fashion shows. The tempo and energy of Response 2 better match the model's confident stride and the overall aesthetic.
- The first audio track is more suitable for 

SCORE_PROMPT = SYS_SCORE_PROMPT
SCORE_PROMPT += f"\n\nHere is the scoring rule:\n{OVERALL_SCORE_SCORING_PROMPT}"
print(SCORE_PROMPT)

In [154]:
score_content_top1 = score_content('image', input, music_top1)
scores, comments = run_score(client, GEMINI_2_5_PRO, score_content_top1, SCORE_PROMPT, 5)
score_overall = majority_score(scores)
score_overall

(5.0, {5: 5})

In [142]:
score_content_top5 = score_content('image', input, music_top5)
scores, comments = run_score(client, GEMINI_2_5_PRO, score_content_top5, SCORE_PROMPT, 5)
score_overall = majority_score(scores)
score_overall

(1.0, {1: 5})